# Создание моделей прогнозирования

In [9]:
from text_metrics.processor import flat_metrics
from predictors.binary_classif.multi import CascadeClassifier, FlatClassifier
from predictors.binary_classif.predictor import Predictor
from predictors.serialization import load_model, save_model
from predictors.transformers import (
    BinarizationRule, ModelWrapper,
    CompositionTransformer, ApplyListTransformer, DataFrameTransformer,
    DefaultValueTransformer, RenameColumnTransformer, BinarizationTransformer,
    DropColumnTransformer
)
from pathlib import Path


## Пайплайн предобработки метрик


In [10]:
metrics_cols = ['text__symbols_length', 'text__sentences_count',
       'text__long_sentences_proportion', 'text__unpopular_words_proportion',
       'text__obscene_proportion', 'text__rare_words_proportion',
       'text__type_token_ratio', 'text__normalized_shannon_entropy',
       'text__mean_words_length', 'text__long_words_proportion',
       'text__lexicon_size', 'text__lexical_diversity_per_sentence',
       'entities__mean_usage_QUOTE', 'entities__mean_usage_NUMBER',
       'entities__mean_usage_MEAS', 'entities__mean_usage_ENUM',
       'entities__mean_usage_FOREIGN', 'entities__breadth_of_use_entities',
       'pos__index_of_formality', 'pos__breadth_of_use_pos',
       'pos__pair_of_adv_per_sentence', 'pos__mean_usage_NOUN',
       'pos__mean_usage_PRON', 'pos__mean_usage_ADV', 'pos__mean_usage_ADP',
       'pos__mean_usage_ADJ', 'pos__mean_usage_SCONJ', 'pos__mean_usage_AUX',
       'pos__mean_usage_CCONJ', 'pos__mean_usage_VERB',
       'pos__mean_usage_PROPN', 'pos__mean_usage_DET', 'pos__mean_usage_PART',
       "pos__('ADJ-Pos', 'NOUN')", "pos__('PROPN', 'PUNCT')",
       "pos__('PUNCT', 'SCONJ')", "pos__('ADP', 'ADJ-Pos')",
       "pos__('ADP', 'PRON')", "pos__('PUNCT', 'ADV')",
       "pos__('PUNCT', 'PRON')", "pos__('ADJ-Pos', 'PUNCT')",
       "pos__('PUNCT', 'CCONJ')", 'pos__index_of_formality_tuldava',
       'pos__index_of_formality_heylinger', 'punct__breadth_of_use_puncts',
       'punct__mean_usage_:', 'punct__mean_usage_"', 'punct__mean_usage_.',
       'punct__mean_usage_?', 'punct__mean_usage_(', 'punct__mean_usage_,',
       'punct__mean_usage_)', 'punct__mean_usage_-',
       'entities__mean_usage_PUNCEM', 'entities__mean_usage_SMILE',
       'pos__mean_usage_NUM', 'punct__mean_usage_!', 'punct__mean_usage_&',
       'entities__mean_usage_ADDRESS', 'entities__mean_usage_DATE',
       'entities__mean_usage_URL', 'punct__mean_usage_]',
       'punct__mean_usage_[', 'punct__mean_usage_;', 'punct__mean_usage_<',
       'punct__mean_usage_>', 'pos__mean_usage_INTJ', 'punct__mean_usage_/',
       'punct__mean_usage_*', 'punct__mean_usage__', 'punct__mean_usage_~',
       'entities__mean_usage_TAG', 'entities__mean_usage_EMAIL',
       'entities__mean_usage_PHONE', 'entities__mean_usage_POINTER']

rename_mapper = {
    "pos__('ADJ-Pos', 'NOUN')": "pos__ADJPOS_NOUN",
    "pos__('PUNCT', 'SCONJ')": "pos__PUNCT_SCONJ",
    "pos__('ADP', 'ADJ-Pos')": "pos__ADP_ADJPOS",
    "pos__('ADP', 'PRON')": "pos__ADP_PRON",
    "pos__('PUNCT', 'ADV')": "pos__PUNCT_ADV",
    "pos__('ADJ-Pos', 'PUNCT')": "pos__ADJPOS_PUNCT",
    "pos__('PUNCT', 'CCONJ')": "pos__PUNCT_CCONJ",
    "pos__('PUNCT', 'PRON')": "pos__PUNCT_PRON",
    "pos__('PROPN', 'PUNCT')": "pos__PROPN_PUNCT",
    'punct__mean_usage_.': "punct__mean_usage_dot",
    'punct__mean_usage_,': "punct__mean_usage_comma",
    'punct__mean_usage_:': "punct__mean_usage_colon",
    'punct__mean_usage_"': "punct__mean_usage_quote",
    'punct__mean_usage_?': "punct__mean_usage_question",
    'punct__mean_usage_(': "punct__mean_usage_openparen",
    'punct__mean_usage_)': "punct__mean_usage_closeparen",
    'punct__mean_usage_-': "punct__mean_usage_hyphen",
    'punct__mean_usage_!': "punct__mean_usage_exclamation",
    'punct__mean_usage_&': "punct__mean_usage_ampersand",
    'punct__mean_usage_]': "punct__mean_usage_closebracket",
    'punct__mean_usage_[': "punct__mean_usage_openbracket",
    'punct__mean_usage_;': "punct__mean_usage_semicolon",
    'punct__mean_usage_<': "punct__mean_usage_lessthan",
    'punct__mean_usage_>': "punct__mean_usage_greaterthan",
    'punct__mean_usage_/': "punct__mean_usage_slash",
    'punct__mean_usage_*': "punct__mean_usage_asterisk",
    'punct__mean_usage__': "punct__mean_usage_underscore",
    'punct__mean_usage_~': "punct__mean_usage_tilde"
}

combo_features = { 
    "pos__pair_of_adv_per_sentence",
    "pos__mean_usage_AUX",
    "pos__mean_usage_NUM",
    "pos__PROPN_PUNCT",
    "pos__PUNCT_PRON",
    "punct__mean_usage_colon",
    "punct__mean_usage_question",
    "punct__mean_usage_openparen",
    "punct__mean_usage_closeparen",
    "punct__mean_usage_hyphen",
    "punct__mean_usage_exclamation",
    "entities__mean_usage_NUMBER",
    "entities__mean_usage_QUOTE",
    "entities__mean_usage_PUNCEM",
    "entities__mean_usage_ADDRESS",
}

typical_ranges = {
    "entities__mean_usage_MEAS": { 
        "min": -0.000005, 
        "max": 0.000005
    },
    "entities__mean_usage_ENUM": { 
        "min": -0.002222, 
        "max": 0.002222
    },
    "entities__mean_usage_FOREIGN": { 
        "min": -0.004762, 
        "max":0.004762
    },
    "pos__breadth_of_use_pos": { 
        "min": 0.806250, 
        "max":0.818750
    },
    "pos__index_of_formality_tuldava": { 
        "min": 98.449087, 
        "max": 101.550913
    },
    "entities__mean_usage_SMILE": { 
        "min": -0.000005, 
        "max":0.000005
    },
    "entities__mean_usage_DATE": { 
        "min": -0.001010, 
        "max":0.001010
    }
}


combo_rules = [ BinarizationRule(col, f"{col}_used", gt_then=0) for col in combo_features ]
typical_rules = [ BinarizationRule(col, f"{col}_is_typical", between=[constraints['min'], constraints['max']]) for col, constraints in typical_ranges.items() ]
remove_typical_cols = [ col for col in typical_ranges ]

In [11]:
transformer = CompositionTransformer(
    [
        ApplyListTransformer(flat_metrics),
        DataFrameTransformer(metrics_cols),
        DefaultValueTransformer(0),
        RenameColumnTransformer(rename_mapper),
        BinarizationTransformer(combo_rules + typical_rules),
        DropColumnTransformer(remove_typical_cols)
    ]
)

## Загрузка моделей и создание предикторов

In [12]:
def load_flat(name):
    return load_model(Path(f"../models/{name}_flat_best/model.pkl"))
    
def load_cascade(name):
    return load_model(Path(f"../models/cascade_{name}/model.pkl"))

In [13]:
cascades = [ 'svc_poly_svc_rbf', 'svc_poly_lr', 'svc_poly_gbm' ]
flats = [ 'svc_rbf', 'svc_poly', 'svc_linear', 'mlp' ]

cascade_models = [ load_cascade(name) for name in cascades ]
flat_models = [ load_flat(name) for name in flats ]

In [14]:
predictors_dir = Path("../predictors")
predictors_dir.mkdir(parents=True, exist_ok=True)

In [15]:
def save_predictors(names, models, prefix, cls_clf):
    for name, model in zip(names, models):
        model_wrapper = ModelWrapper(transformer, model)
        clf = cls_clf(model_wrapper)
        predictor = Predictor(clf, thresholds=[0.5, 0.5], labels=[["woman", "man"], ["young", "old"]]) 
        save_model(predictors_dir, predictor, f"{prefix}_{name}")
    

In [16]:
save_predictors(flats, flat_models, "flat", FlatClassifier)
save_predictors(cascades, cascade_models, "cascade", CascadeClassifier)